# Week 11 guided lab · Predicting PM2.5, step by step

**CMT124** · two-hour **guided** lab

**The question:** how well can we predict a day's PM2.5 from weather and traffic, which measurements do we really need, and where should we **not** trust the prediction?

You'll build one complete workflow, the one from today's lectures, on the same synthetic air-quality data: clean → split → baseline → model → residuals → better model → feature selection → comparison → prediction.

### How this guided lab works

As in Weeks 6 and 7, **we work through it together, one step at a time.**

| Each step has | What you do |
|---|---|
| **Do** | one operation, in the empty code cell |
| **Checkpoint** | run the checkpoint cell and compare what it prints (or draws) with your neighbours |
| **Think** | a question we answer together; jot down the answer in the notes cell |

* Only one step is open at a time. Finished early? Read the next **Think** question and prepare an answer.
* A checkpoint that prints `✔` means your result matches. An `AssertionError` tells you what to look at.
* Stuck? Under the key steps there is a cell headed *"If your result does not match the checkpoint, run this cell to continue."* Run it and carry on with everyone else; come back to your own version later.
* Everything uses this week's tools: Boolean filtering, `train_test_split`, `LinearRegression`, `RandomForestRegressor`, `.predict`, residuals, MAE / RMSE / R² and `permutation_importance`.

| Step | One operation | Checkpoint |
|---|---|---|
| 1 | find the impossible PM2.5 values | how many, and what they have in common |
| 2 | clean the data and split it | training and test shapes |
| 3 | a baseline that knows nothing | its MAE, RMSE and R² |
| 4 | a linear model on two features | scores and slopes |
| 5 | the residuals of that model | a residual plot |
| 6 | all four features: linear regression and a random forest | two rows of scores |
| 7 | which features matter? | permutation importance |
| 8 | refit with the selected features | a table of four models |
| 9 | predictions for three new days | three numbers to judge |

### Setup

Run this cell first. It loads the libraries, defines `make_air_quality()` (the generator from the lectures) and a helper `scores(y_true, y_pred)` that returns MAE, RMSE and R². It also makes the data, `air`.

*The data are synthetic: 5,000 days made by a formula plus random noise for teaching, not measurements from a real site.*

In [ ]:
#@title Setup: run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance


def make_air_quality(n=5000, seed=3622626236, temp_mean=20):
    """Synthetic daily records at one roadside site: weather, traffic and PM2.5 (µg/m³)."""
    rng = np.random.RandomState(seed)
    temperature = rng.normal(temp_mean, 5, n)            # °C
    humidity = rng.uniform(30, 90, n)                    # %
    wind_speed = rng.exponential(2, n)                   # m/s
    traffic_intensity = rng.uniform(0, 100, n)           # arbitrary units
    pm25 = (5 + 0.4 * traffic_intensity
            + 0.1 * traffic_intensity * (temperature - 20)
            + 0.3 * (30 - np.clip(wind_speed, 0, 30))
            + 0.1 * (humidity - 50)
            - 0.3 * (temperature - 20))
    noise = 0.5 * rng.normal(0, 10, n) + 0.5 * rng.normal(5, 5, n)
    return pd.DataFrame({"temperature": temperature, "humidity": humidity,
                         "wind_speed": wind_speed, "traffic_intensity": traffic_intensity,
                         "pm25": pm25 + noise})


def scores(y_true, y_pred):
    """MAE, RMSE (both in µg/m³) and R² of a set of predictions."""
    return {"MAE": round(mean_absolute_error(y_true, y_pred), 2),
            "RMSE": round(mean_squared_error(y_true, y_pred) ** 0.5, 2),
            "R2": round(r2_score(y_true, y_pred), 3)}


air = make_air_quality()
print(air.shape)

## Step 1 · Find the impossible values

**Step 1 of 9** · ●○○○○○○○○

In the lecture we counted the negative PM2.5 values. Now we ask **where** they occur.

**Do:** keep the rows of `air` whose `pm25` is below zero, as `negative`.

In [ ]:
# STUDENT TASK
# Task: negative = the rows of air with pm25 < 0
# Hint: Boolean filtering, as in Week 6: air[air["pm25"] < 0]
# your code here


### Checkpoint 1

In [ ]:
# Checkpoint 1: how many negative days, and how do they differ from all days?
print(len(negative), "days with negative PM2.5")
comparison = pd.DataFrame({"negative days": negative.mean(), "all days": air.mean()}).round(1)
print(comparison)
assert len(negative) == 266, "expected 266 rows: keep the rows where pm25 is below 0"
print("✔ Step 1")

### Think 1

Compare the two columns. What do the negative days have in common? If we **remove** them, which kind of day will be missing from our data, and what might that mean for a model trained on what's left?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 2 opens.

## Step 2 · Clean the data and split it

**Step 2 of 9** · ●●○○○○○○○

**Do:**
1. `air_clean` = the rows of `air` with `pm25` **at least** zero;
2. `X` = the four feature columns `["temperature", "humidity", "wind_speed", "traffic_intensity"]` of `air_clean`; `y` = its `pm25` column;
3. split into `X_train, X_test, y_train, y_test` with `test_size=0.2` and `random_state=42`.

In [ ]:
# STUDENT TASK
# Task: air_clean, X (four features), y (pm25), then an 80/20 split with random_state=42
# Hint: X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# your code here


### Checkpoint 2

In [ ]:
# Checkpoint 2: shapes, and how many cold, busy days are left
print("training:", X_train.shape, "  test:", X_test.shape)
cold_busy = (air["temperature"] < 10) & (air["traffic_intensity"] > 70)
print("cold, busy days (below 10 °C, traffic above 70) before cleaning:", cold_busy.sum())
print("... after cleaning:", ((air_clean["temperature"] < 10) & (air_clean["traffic_intensity"] > 70)).sum())
assert X_train.shape == (3787, 4) and X_test.shape == (947, 4), "expected (3787, 4) and (947, 4): check the filter (>= 0) and test_size"
print("✔ Step 2")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
features = ["temperature", "humidity", "wind_speed", "traffic_intensity"]
air_clean = air[air["pm25"] >= 0]
X = air_clean[features]
y = air_clean["pm25"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Think 2

Every one of the 35 cold, busy days has gone. We remove the bad rows **before** splitting. Would it be wrong to split first and then clean only the training set?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 3 opens.

## Step 3 · A baseline that knows nothing

**Step 3 of 9** · ●●●○○○○○○

Before we fit anything, how good is a "model" that ignores the features and predicts the **training mean** for every test day? Any real model must beat it.

**Do:** make `baseline_pred`, an array with one value per test day, all equal to the mean of `y_train`.

In [ ]:
# STUDENT TASK
# Task: baseline_pred = the mean of y_train, repeated once for every test day
# Hint: np.full(len(y_test), y_train.mean())
# your code here


### Checkpoint 3

In [ ]:
# Checkpoint 3: the baseline's scores
print("training mean:", round(y_train.mean(), 2), "µg/m³")
print("baseline:", scores(y_test, baseline_pred))
assert len(baseline_pred) == 947, "one prediction per test day"
assert abs(scores(y_test, baseline_pred)["MAE"] - 21.16) < 0.01, "use the mean of y_train (not y_test)"
print("✔ Step 3")

### Think 3

The baseline's R² is about **0**. Why is that exactly what R² should say about this "model"? And why do we use the mean of the **training** days, not of the test days?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 4 opens.

## Step 4 · A linear model on two features

**Step 4 of 9** · ●●●●○○○○○

**Do:** fit a `LinearRegression` called `lin2` on the training data using only `["temperature", "traffic_intensity"]`, then predict the test days as `pred_lin2`.

In [ ]:
# STUDENT TASK
# Task: lin2 = LinearRegression fitted on temperature and traffic_intensity; pred_lin2 = its test predictions
# Hint: two = ["temperature", "traffic_intensity"]; fit on X_train[two], predict X_test[two]
# your code here


### Checkpoint 4

In [ ]:
# Checkpoint 4: scores and slopes
print("linear, 2 features:", scores(y_test, pred_lin2))
print(f"slopes: temperature {lin2.coef_[0]:.3f}, traffic_intensity {lin2.coef_[1]:.3f};  intercept {lin2.intercept_:.2f}")
assert abs(scores(y_test, pred_lin2)["R2"] - 0.701) < 0.002, "fit on the training set with the two features, predict the test set"
print("✔ Step 4")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
two = ["temperature", "traffic_intensity"]
lin2 = LinearRegression().fit(X_train[two], y_train)
pred_lin2 = lin2.predict(X_test[two])

### Think 4

Put the two slopes into words, with units. The temperature slope is positive: does that show that warmer weather **causes** more PM2.5?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 5 opens.

## Step 5 · What the residuals say

**Step 5 of 9** · ●●●●●○○○○

**Do:** compute the test residuals of `lin2` as `res` (actual − predicted), then draw a scatter plot of `res` (*y*) against `pred_lin2` (*x*), with a horizontal line at 0 and axis labels.

In [ ]:
# STUDENT TASK
# Task: res = residuals of lin2 on the test set; scatter of res against pred_lin2 with a line at 0
# Hint: res = y_test - pred_lin2; plt.scatter(pred_lin2, res, s=5, alpha=0.4); plt.axhline(0, color="black")
# your code here


### Checkpoint 5

In [ ]:
# Checkpoint 5: a few numbers to go with your plot
print("mean residual:", round(res.mean(), 2), "µg/m³")
print("largest under-prediction:", round(res.max(), 1), "  largest over-prediction:", round(res.min(), 1))
print("test days predicted BELOW zero:", (pred_lin2 < 0).sum())
assert abs(res.mean() - (-0.62)) < 0.01, "res = y_test - pred_lin2 (actual minus predicted)"
print("✔ Step 5")

### Think 5

**Halfway question.** The mean residual is close to zero, yet the plot is clearly not a shapeless band. Describe the pattern. What does it tell you about the straight-line model, and what would you try next? What do you make of the predictions below zero?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 6 opens.

## Step 6 · All four features: linear regression and a random forest

**Step 6 of 9** · ●●●●●●○○○

**Do:** using **all four** features (`X_train`, `X_test`):
1. fit a `LinearRegression` called `lin4` and predict the test set as `pred_lin4`;
2. fit a `RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)` called `forest4` and predict the test set as `pred_forest4`.

In [ ]:
# STUDENT TASK
# Task: lin4 and forest4 on all four features; pred_lin4 and pred_forest4 on the test set
# Hint: forest4 = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1).fit(X_train, y_train)
# your code here


### Checkpoint 6

In [ ]:
# Checkpoint 6: two rows of scores
print(pd.DataFrame({"linear, 2 features": scores(y_test, pred_lin2),
                    "linear, 4 features": scores(y_test, pred_lin4),
                    "forest, 4 features": scores(y_test, pred_forest4)}).T)
assert abs(scores(y_test, pred_lin4)["R2"] - 0.707) < 0.002, "lin4: LinearRegression on all four features"
assert abs(scores(y_test, pred_forest4)["R2"] - 0.944) < 0.005, "forest4: n_estimators=200, random_state=42, all four features"
print("✔ Step 6")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
lin4 = LinearRegression().fit(X_train, y_train)
pred_lin4 = lin4.predict(X_test)
forest4 = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1).fit(X_train, y_train)
pred_forest4 = forest4.predict(X_test)

### Think 6

Adding humidity and wind speed to the linear model barely changed anything (R² 0.701 → 0.707). Changing the **model** from linear to a random forest, with the same four features, changed a lot (0.707 → 0.944). Why?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 7 opens.

## Step 7 · Which features matter?

**Step 7 of 9** · ●●●●●●●○○

**Do:** compute the permutation importance of `forest4` on the **test** set with `n_repeats=5` and `random_state=42`, and store the mean importances as a pandas Series `importance` with the feature names as index, sorted from smallest to largest.

In [ ]:
# STUDENT TASK
# Task: importance = mean permutation importance of forest4 per feature (test set), sorted
# Hint: result = permutation_importance(forest4, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
# your code here


### Checkpoint 7

In [ ]:
# Checkpoint 7: the importances, next to each feature's correlation with PM2.5 in the training set
correlation = X_train.join(y_train).corr()["pm25"].drop("pm25")
print(pd.DataFrame({"permutation importance": importance, "correlation": correlation})
      .sort_values("permutation importance", ascending=False).round(3))
importance.plot.barh(xlabel="drop in R² when the feature is shuffled")
plt.show()
assert set(importance.index[-2:]) == {"temperature", "traffic_intensity"}, "sort the Series from smallest to largest"
print("✔ Step 7")

### Think 7

Wind speed has almost **zero** correlation with PM2.5 and almost **zero** importance. Does that prove that wind has no effect on PM2.5 in these data? What does a low importance actually tell you?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 8 opens.

## Step 8 · Refit with the selected features

**Step 8 of 9** · ●●●●●●●●○

**Do:** fit a random forest called `forest2` (same settings as `forest4`) on the two selected features `["temperature", "traffic_intensity"]` only, and predict the test set as `pred_forest2`.

In [ ]:
# STUDENT TASK
# Task: forest2 = the same random forest on temperature and traffic_intensity only; pred_forest2 = its test predictions
# Hint: use the list two from Step 4: forest2.fit(X_train[two], y_train)
# your code here


### Checkpoint 8

In [ ]:
# Checkpoint 8: four models, one table
table = pd.DataFrame({"linear, 2 features": scores(y_test, pred_lin2),
                      "linear, 4 features": scores(y_test, pred_lin4),
                      "forest, 4 features": scores(y_test, pred_forest4),
                      "forest, 2 features": scores(y_test, pred_forest2)}).T
print(table)
assert abs(scores(y_test, pred_forest2)["R2"] - 0.933) < 0.005, "forest2: same settings, features temperature and traffic_intensity"
print("✔ Step 8")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
two = ["temperature", "traffic_intensity"]
forest2 = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1).fit(X_train[two], y_train)
pred_forest2 = forest2.predict(X_test[two])

### Think 8

Did the smaller forest lose accuracy? If you had to **run this model every day** at a real roadside site, which of the four would you choose, and why?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 9 opens.

## Step 9 · Predictions for three new days

**Step 9 of 9** · ●●●●●●●●●

A model is built to predict days it has never seen. Here are three:

| Day | Temperature (°C) | Traffic intensity |
|---|---|---|
| A: cold and busy | 5 | 90 |
| B: typical | 20 | 50 |
| C: warm and busy | 28 | 90 |

**Do:** put the three days in a DataFrame `new_days` with the columns `temperature` and `traffic_intensity` (in that order), and predict them with `forest2` as `pred_new`.

In [ ]:
# STUDENT TASK
# Task: new_days (3 rows, columns temperature and traffic_intensity); pred_new = forest2's predictions
# Hint: pd.DataFrame({"temperature": [5, 20, 28], "traffic_intensity": [90, 50, 90]})
# your code here


### Checkpoint 9

In [ ]:
# Checkpoint 9: the forest's predictions, and the straight line's for comparison
print(pd.DataFrame({"forest, 2 features": pred_new.round(1),
                    "linear, 2 features": lin2.predict(new_days).round(1)},
                   index=["A: cold and busy", "B: typical", "C: warm and busy"]))
print("coldest training day:", round(X_train["temperature"].min(), 1), "°C")
assert pred_new.shape == (3,), "three predictions, one per day"
print("✔ Step 9")

### Think 9

Which of the three predictions do you trust **least**, and why? Would you use this model to warn the public in **another city**, or in **another year**? What would you want to check first?

*Notes from the discussion:*

## Optional: if you finish early

None of this is needed in Week 12.

**A · Watch over-fitting.** Fit a `DecisionTreeRegressor(max_depth=d, random_state=42)` on `X_train` for `d` in `[1, 2, 4, 6, 8, 12, 20]`, and print the training and test RMSE for each depth. Where is the test RMSE lowest? What happens to the training RMSE?

**B · A better feature for the linear model.** Add a column `temp_x_traffic = temperature * traffic_intensity` to copies of `X_train[two]` and `X_test[two]`, refit `LinearRegression`, and compare its `scores` with `forest2`.

**C · How stable is the score?** Use `cross_val_score(forest, X_train[two], y_train, cv=5, scoring="r2")` (from `sklearn.model_selection`) with a 100-tree forest. How much do the five R² values vary?

In [ ]:
# your code here

## Wrap-up

* **Clean first**, and record what cleaning removed: here, every cold, busy day.
* Always compare with a **baseline**; read **residuals**, not just scores.
* A better model **form** (random forest) gained more than more features; **feature selection** kept two features for almost all the accuracy.
* Predictions are trustworthy only **inside** the conditions the model was trained on.

**Next week** the same data get a different question: *is this a high-pollution day?*